Segment customers based on behavior and demographics

Key Features:
Perform clustering using tools like Python (scikit-learn) or Power BI
Analyze purchase patterns and customer preferences
Visualize segments and key characteristics
Expected Outcome:
Gain experience in customer analytics, segmentation, and targeted insights

Install & Import Everything

In [17]:
!pip install scikit-learn plotly --quiet

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings("ignore")

print("✅ All libraries loaded!")

✅ All libraries loaded!


Load & Explore the Data

In [18]:
df = pd.read_csv("Mall_Customers.csv")

# Rename columns for easier use
df.columns = ["CustomerID", "Gender", "Age", "Annual_Income", "Spending_Score"]

print("Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

Shape: (200, 5)

First 5 rows:


,CustomerID,Gender,Age,Annual_Income,Spending_Score
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40


Check Data Quality

In [19]:
print("=== Data Info ===")
print(df.dtypes)
print("\n=== Missing Values ===")
print(df.isnull().sum())
print("\n=== Basic Statistics ===")
df.describe().round(2)

=== Data Info ===
CustomerID         int64
Gender            object
Age                int64
Annual_Income      int64
Spending_Score     int64
dtype: object

=== Missing Values ===
CustomerID        0
Gender            0
Age               0
Annual_Income     0
Spending_Score    0
dtype: int64

=== Basic Statistics ===


,CustomerID,Age,Annual_Income,Spending_Score
count,200.00,200.00,200.00,200.00
mean,100.50,38.85,60.56,50.20
std,57.88,13.97,26.26,25.82
min,1.00,18.00,15.00,1.00
25%,50.75,28.75,41.50,34.75
50%,100.50,36.00,61.50,50.00
75%,150.25,49.00,78.00,73.00
max,200.00,70.00,137.00,99.00


Explore Demographics (Charts)

In [20]:
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["Gender Split", "Age Distribution", "Income Distribution"],
    specs=[[{"type": "domain"}, {"type": "xy"}, {"type": "xy"}]]
)

# Gender pie chart
gender_counts = df["Gender"].value_counts()
fig.add_trace(go.Pie(
    labels=gender_counts.index,
    values=gender_counts.values,
    hole=0.4,
    marker_colors=["#378ADD", "#F472B6"]
), row=1, col=1)

# Age histogram
fig.add_trace(go.Histogram(
    x=df["Age"], nbinsx=15,
    marker_color="#1D9E75",
    name="Age"
), row=1, col=2)

# Income histogram
fig.add_trace(go.Histogram(
    x=df["Annual_Income"], nbinsx=15,
    marker_color="#EF9F27",
    name="Income"
), row=1, col=3)

fig.update_layout(
    title="Customer Demographics Overview",
    title_font_size=18,
    plot_bgcolor="white",
    height=380,
    showlegend=False
)
fig.show()

Spending Score vs Income (Raw Data)

In [21]:
fig = px.scatter(df, x="Annual_Income", y="Spending_Score",
    color="Gender", size="Age",
    title="💰 Spending Score vs Annual Income (Before Clustering)",
    labels={"Annual_Income":"Annual Income (k$)", "Spending_Score":"Spending Score (1-100)"},
    color_discrete_map={"Male":"#378ADD", "Female":"#F472B6"},
    hover_data=["CustomerID","Age"])

fig.update_layout(plot_bgcolor="white", title_font_size=16)
fig.show()

Find the Best Number of Clusters (Elbow Method)

In [22]:
X = df[["Annual_Income", "Spending_Score"]]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

inertia = []
silhouette = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)
    silhouette.append(silhouette_score(X_scaled, kmeans.labels_))

fig = make_subplots(rows=1, cols=2,
    subplot_titles=["Elbow Method (find the bend)", "Silhouette Score (higher = better)"])

fig.add_trace(go.Scatter(x=list(K_range), y=inertia,
    mode="lines+markers", line=dict(color="#378ADD", width=3),
    marker=dict(size=8), name="Inertia"), row=1, col=1)

fig.add_trace(go.Scatter(x=list(K_range), y=silhouette,
    mode="lines+markers", line=dict(color="#1D9E75", width=3),
    marker=dict(size=8), name="Silhouette"), row=1, col=2)

fig.add_vline(x=5, line_dash="dash", line_color="red",
    annotation_text="Best K=5", row=1, col=1)
fig.add_vline(x=5, line_dash="dash", line_color="red", row=1, col=2)

fig.update_layout(title="📊 Finding the Optimal Number of Clusters",
    title_font_size=16, plot_bgcolor="white", height=380, showlegend=False)
fig.show()
print("✅ Best number of clusters = 5")

✅ Best number of clusters = 5


Run K-Means Clustering (The Main ML Step)

In [23]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X_scaled)

# Give each cluster a meaningful business name
cluster_names = {
    0: "💎 High Income, Low Spenders",
    1: "🌟 High Income, High Spenders",
    2: "💰 Average Income, Average Spenders",
    3: "🎯 Low Income, High Spenders",
    4: "📉 Low Income, Low Spenders"
}

# Auto-assign names based on income & spending averages
cluster_stats = df.groupby("Cluster")[["Annual_Income","Spending_Score"]].mean()
print("Cluster Averages:")
print(cluster_stats.round(1))
df["Segment"] = df["Cluster"].map(cluster_names)

print("\n✅ Clustering complete! Customer segments assigned.")

Cluster Averages:
         Annual_Income  Spending_Score
Cluster                               
0                 55.3            49.5
1                 86.5            82.1
2                 25.7            79.4
3                 88.2            17.1
4                 26.3            20.9

✅ Clustering complete! Customer segments assigned.


Visualize the Clusters (Main Result Chart)

In [24]:
colors = ["#378ADD","#1D9E75","#EF9F27","#D85A30","#534AB7"]

fig = px.scatter(df, x="Annual_Income", y="Spending_Score",
    color="Segment",
    title="🎯 Customer Segments — K-Means Clustering Result",
    labels={"Annual_Income":"Annual Income (k$)","Spending_Score":"Spending Score (1-100)"},
    color_discrete_sequence=colors,
    size_max=14,
    hover_data=["Age","Gender","CustomerID"])

# Add cluster center markers
centers_original = scaler.inverse_transform(kmeans.cluster_centers_)
fig.add_trace(go.Scatter(
    x=centers_original[:,0], y=centers_original[:,1],
    mode="markers", marker=dict(symbol="x", size=16, color="black", line=dict(width=2)),
    name="Cluster Centers"))

fig.update_layout(plot_bgcolor="white", title_font_size=18, height=520)
fig.update_traces(marker=dict(size=10, opacity=0.85))
fig.show()

Segment Size & Profile

In [25]:
segment_summary = df.groupby("Segment").agg(
    Count=("CustomerID","count"),
    Avg_Age=("Age","mean"),
    Avg_Income=("Annual_Income","mean"),
    Avg_Spending=("Spending_Score","mean")
).round(1).reset_index()

print("=== Customer Segment Profiles ===")
print(segment_summary.to_string(index=False))

# Bar chart of segment sizes
fig = px.bar(segment_summary, x="Segment", y="Count",
    color="Segment",
    title="👥 Number of Customers per Segment",
    color_discrete_sequence=["#378ADD","#1D9E75","#EF9F27","#D85A30","#534AB7"],
    text="Count")
fig.update_traces(textposition="outside")
fig.update_layout(plot_bgcolor="white", showlegend=False,
    title_font_size=16, xaxis_tickangle=-20)
fig.show()

=== Customer Segment Profiles ===
                           Segment  Count  Avg_Age  Avg_Income  Avg_Spending
      🌟 High Income, High Spenders     39     32.7        86.5          82.1
       🎯 Low Income, High Spenders     35     41.1        88.2          17.1
       💎 High Income, Low Spenders     81     42.7        55.3          49.5
💰 Average Income, Average Spenders     22     25.3        25.7          79.4
        📉 Low Income, Low Spenders     23     45.2        26.3          20.9


3D Cluster Visualization

In [26]:
# Better alternative to 3D — works everywhere, looks great
fig = px.scatter(df,
    x="Annual_Income",
    y="Spending_Score",
    color="Segment",
    facet_col="Gender",
    title="🎯 Customer Segments by Gender — Income vs Spending",
    labels={
        "Annual_Income": "Annual Income (k$)",
        "Spending_Score": "Spending Score (1-100)"
    },
    color_discrete_sequence=["#378ADD","#1D9E75","#EF9F27","#D85A30","#534AB7"],
    size_max=14,
    hover_data=["Age", "CustomerID"])

fig.update_traces(marker=dict(size=10, opacity=0.85))
fig.update_layout(
    plot_bgcolor="white",
    title_font_size=18,
    height=500)
fig.show()
print("✅ Chart rendered successfully!")

✅ Chart rendered successfully!


Business Insights Summary

In [27]:
insights = """
╔══════════════════════════════════════════════════════════════╗
║           CUSTOMER SEGMENTATION — BUSINESS INSIGHTS          ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  🌟 HIGH INCOME + HIGH SPENDERS  → VIP Customers            ║
║     Target with: Premium products, loyalty rewards           ║
║                                                              ║
║  💎 HIGH INCOME + LOW SPENDERS   → Untapped Potential        ║
║     Target with: Exclusive offers, personalized deals        ║
║                                                              ║
║  🎯 LOW INCOME + HIGH SPENDERS   → Impulsive Buyers          ║
║     Target with: EMI offers, budget-friendly combos          ║
║                                                              ║
║  📉 LOW INCOME + LOW SPENDERS    → Price-Sensitive Group     ║
║     Target with: Heavy discounts, value deals                ║
║                                                              ║
║  💰 AVERAGE INCOME + AVERAGE     → Middle Market             ║
║     Target with: Seasonal offers, mid-range products         ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
"""
print(insights)


╔══════════════════════════════════════════════════════════════╗
║           CUSTOMER SEGMENTATION — BUSINESS INSIGHTS          ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  🌟 HIGH INCOME + HIGH SPENDERS  → VIP Customers            ║
║     Target with: Premium products, loyalty rewards           ║
║                                                              ║
║  💎 HIGH INCOME + LOW SPENDERS   → Untapped Potential        ║
║     Target with: Exclusive offers, personalized deals        ║
║                                                              ║
║  🎯 LOW INCOME + HIGH SPENDERS   → Impulsive Buyers          ║
║     Target with: EMI offers, budget-friendly combos          ║
║                                                              ║
║  📉 LOW INCOME + LOW SPENDERS    → Price-Sensitive Group     ║
║     Target with: Heavy discounts, value deals                ║
║                            